# **(Co_Risk_analyzer)**

## Objectives

# ETL & Analysis: risk hypotheses from clinical features
This notebook performs an ETL pipeline, feature engineering, and visualizations using only pandas and numpy. It evaluates these hypotheses:
- High BMI is associated with higher `Heart_Attack_Risk_Percentage`.
- A positive `Family_History_CVD` is associated with higher `Heart_Attack_Risk_Percentage`.
- High BMI and `Family_History_CVD` together show higher heart attack risk than either factor alone.
- Current `Smoking_Status` is associated with higher `Heart_Attack_Risk_Percentage`.
- High `Systolic_BP` (>= 140) is associated with higher `Mortality_Risk_Percentage`.
- Low `Physical_Activity_Level` is associated with higher `Heart_Attack_Risk_Percentage` and `Mortality_Risk_Percentage`.
- `Diabetes` present is associated with higher `Mortality_Risk_Percentage`.
- Low `Medication_Adherence_Percentage` is associated with higher `Mortality_Risk_Percentage`.


---

## Data Source

The primary dataset for this project is Patient comorbidity risk assessment data collected from Kaggle intended for research purposes https://www.kaggle.com/datasets/velvetcrystal/patient-comorbidity-risk-assessment-dataset


In [ ]:
# Importing necessary python packages for ETL And visualization purpose.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
print('imports ok')

* To make the notebook work on different devices, we first identify the current working folder and then change it to its parent folder so that file paths are easier to access consistently..

# Change working directory

* We access the current directory with os.getcwd()
We want to make the parent of the current directory the new current directory
* Function find_project_root sets the current directory by applying for loop and if conditions to avoid ris of re-runs setting inappropiate dirctory path.
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
import os
from pathlib import Path


def find_project_root(start=None):
    if start is None:
        start = Path.cwd().resolve()

    candidates = []
    for candidate in [start, *start.parents]:
        candidates.append(candidate)

    # Also try the project folder directly if it exists on this machine.
    known_project = Path(r"C:/Users/User/Desktop/Data Analysis with AI Course/Code Institute/vscode-projects/CoRiskAnalyzer/CoRiskAnalyzer")
    if known_project.exists():
        candidates.append(known_project)

    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)

        if (candidate / "Datasets").exists():
            return candidate

        if candidate.name.lower() == "jupyter_notebooks" and (candidate.parent / "Datasets").exists():
            return candidate.parent

    return start


project_dir = find_project_root()
os.chdir(project_dir)
print("Project directory:", project_dir)
current_dir = os.getcwd()
print("Current directory:", current_dir)


# Loading the data to a Data Framework by reading from csv file.

In [ ]:
df = pd.read_csv(current_dir + '/Datasets/Patient_Comorbidity_Risk_Assessment_Dataset.csv')
print('Dataframe loaded successfully')

# overview of rows and columns of the data
# Gettingt the dtypes of columns which can be numerical or catogorical

In [ ]:
# displaying the first 5 rows
df.head()

---

In [ ]:
#discovering the shape of the dataset and the column names
df.shape
df.info()

In [ ]:
# view all columns in data set
df.columns

In [ ]:
# check data types of each column
df.dtypes

In [ ]:
# check the summary statistics of the dataset
df.describe()

---

# Section 1.2 Transforming the raw, Uncleaned data : Data Cleaning & Pre-processing

This Section will clean the data by identifying the missing values, duplicates and invalid values and refining them to shape the dataframe ready for anlysis.
checks to perform suitable dtypes for each attribute is available for further analysis.

In [ ]:
# view all columns names in dataset
df.columns

In [ ]:
#getting the count of missing values in each column
df.isnull().sum()

In [ ]:
# check data types of each column
df.dtypes

In [ ]:
# Identifying the missing values in the dataset
df.isnull().sum()

## # ETL: basic cleaning and type conversions for the dataset for Hypothesis testing and visualization purposes.

#Addressing the Numeric columns

In [ ]:
# converting the data types of the columns to appropriate types and filling missing values for numeric columns with mean of the column
num_cols = ['BMI','Cholesterol_Level', 'Systolic_BP', 'Diastolic_BP','Resting_Heart_Rate','HbA1c','Medication_Adherence_Percentage']

#check the outliers in the numeric columns using boxplots
for c in num_cols:
    df[c].plot(kind='box')
    plt.title(f'Boxplot for {c}')
    plt.ylabel(c)
    plt.show()

# As the number of outliers is high, we will use the median to fill the missing values for the numeric columns. The median is less affected by outliers and will provide a more robust estimate of the central tendency of the data.
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df.fillna({c: df[c].median() for c in num_cols}, inplace=True)


# Addressing the catogorical columns

In [ ]:
# Smoking status fill and simplifying the missing values to 'Unknown'
if 'Smoking_Status' in df.columns:
    df['Smoking_Status'] = df['Smoking_Status'].fillna('Unknown')

# Physical Activity Level fill and simplifying the missing values to'Unknown'
if 'Physical_Activity_Level' in df.columns:
    df['Physical_Activity_Level'] = df['Physical_Activity_Level'].fillna('Unknown')

# Create comorbidity_count for each patient by counting the number of comorbidities they have. We will consider the following columns as comorbidities: 'Diabetes', 'Hypertension', 'Heart_Disease', 'Chronic_Kidney_Disease', 'Chronic_Lung_Disease', 'Cancer', 'Stroke', 'Obesity', 'Depression', 'Anxiety'. We will create a new column called 'comorbidity_count' that will contain the count of comorbidities for each patient.
comorbidity_cols = ['Diabetes', 'Hypertension', 'Chronic_Kidney_Disease', 'Liver_Disease']
df['comorbidity_count'] = df[comorbidity_cols].fillna(0).astype(int).sum(axis=1)

# Create risk flags
df['SBP_high'] = (df['Systolic_BP'] > 140).astype(int)
df['Low_Adherence'] = (df['Medication_Adherence_Percentage'] < 80).astype(int)
df.head()

## Featue Engineering 

In [ ]:
# Feature engineering BMI categories and obesity flag
# creating BMI categories column based on WHO classification representing the BMI ranges for underweight, normal weight, overweight and obese. We will create a new column called 'BMI_Category' that will contain the BMI category for each patient based on their BMI value. We will also create a new column called 'Obesity_Flag' that will contain a binary value indicating whether the patient is obese or not based on their BMI value.
def bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif 18.5 <= bmi < 25:
        return 'Normal weight'
    elif 25 <= bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'
df['BMI_Category'] = df['BMI'].apply(bmi_category)
df['Obesity_Flag'] = (df['BMI_Category'] == 'Obese').astype(int)
# creating a new column Obese_x_FH with intraction of Obesity_Flag and Family_History_CVD to identify patients who are obese and have a family history of disease. We will create a new column called 'Obese_x_FH' that will contain a binary value indicating whether the patient is obese and has a family history of disease or not.
df['Obesity_x_FH'] = df['Obesity_Flag'] * df['Family_History_CVD']

#Simple encoding for smoking (current vs Other) with introducing 'Smoking_current' with binary values for current smokers and non-current smokers. We will create a new column called 'Smoking_current' that will contain a binary value indicating whether the patient is a current smoker or not based on their smoking status.
df['Smoking_current'] = (df['Smoking_Status'].str.lower() == 'current').astype(int)
df[['BMI','BMI_Category','Obesity_Flag','Family_History_CVD','Obesity_x_FH']].head()

---

## Visualization and Modeling for Hypothesis testing

###  Hypothesis 1 : High BMI is associated with higher `Heart_Attack_Risk_Percentage`.

In [ ]:
plt.figure(figsize=(7,5))
sns.barplot(x='BMI_Category', y='Heart_Attack_Risk_Percentage', data=df)

###  Hypothesis 2 : A positive `Family_History_CVD` is associated with higher `Heart_Attack_Risk_Percentage`.

In [ ]:
plt.figure(figsize=(7,5))
sns.barplot(x='Family_History_CVD', y='Heart_Attack_Risk_Percentage', data=df)

###  Hypothesis 3 : High BMI and `Family_History_CVD` together show higher heart attack risk than either factor alone.

In [ ]:
plt.figure(figsize=(6,5))

###  Hypothesis 4 :Current `Smoking_Status` is associated with higher `Heart_Attack_Risk_Percentage`.


In [ ]:
df.groupby('Smoking_current')['Heart_Attack_Risk_Percentage'].mean().rename('mean_risk').reset_index()
plt.figure(figsize=(6,5))
sns.boxplot(x='Smoking_current', y='Heart_Attack_Risk_Percentage', data=df)
plt.xticks([0,1], ['Not Current','Current'])
plt.title('Heart Attack Risk by Smoking Status')
plt.show()

###  Hypothesis 5 : High `Systolic_BP` (>= 140) is associated with higher `Mortality_Risk_Percentage`.


In [ ]:
plt.figure(figsize=(6,5))
sns.boxplot(x='SBP_high', y='Mortality_Risk_Percentage', data=df)
plt.xticks([0,1], ['SBP < 140','SBP >= 140'])
plt.title('Mortality Risk by High Systolic BP')
plt.show()

###  Hypothesis 6 : Low `Physical_Activity_Level` is associated with higher `Heart_Attack_Risk_Percentage` and `Mortality_Risk_Percentage`.


In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(x='Physical_Activity_Level', y='Heart_Attack_Risk_Percentage', data=df)
plt.title('Heart Attack Risk by Physical Activity Level')
plt.show()

plt.figure(figsize=(9,5))
sns.barplot(x='Physical_Activity_Level', y='Mortality_Risk_Percentage', data=df)
plt.title('Mortality Risk by Physical Activity Level')
plt.show()

###  Hypothesis 7 :`Diabetes` present is associated with higher `Mortality_Risk_Percentage`.


In [ ]:
plt.figure(figsize=(6,5))
sns.barplot(x='Diabetes', y='Mortality_Risk_Percentage', data=df)
plt.xticks([0,1], ['No Diabetes','Diabetes'])
plt.title('Mortality Risk by Diabetes Presence')
plt.show()

###  Hypothesis 8 :Low `Medication_Adherence_Percentage` is associated with higher `Mortality_Risk_Percentage`.

In [ ]:
plt.figure(figsize=(6,5))
sns.barplot(x='Low_Adherence', y='Mortality_Risk_Percentage', data=df)
plt.xticks([0,1], ['High Adherence','Low Adherence'])
plt.title('Mortality Risk by Medication Adherence')
plt.show()

## Outputs

